# Lab - Entrenamiento de modelo en AWS

Este notebook orquesta dos jobs de SageMaker sobre el dataset Titanic:

1. **Processing Job** (`src/preprocessing.py`): limpieza, feature engineering y split train/validation.
2. **Training Job** (`src/train.py`): entrenamiento de un `RandomForestClassifier` con script personalizado.

Todo el cómputo se delega a instancias efímeras de AWS; este notebook actúa únicamente como orquestador.

## 1. Setup

In [ ]:
import boto3
import sagemaker
from sagemaker import get_execution_role

sess = sagemaker.Session()
role = get_execution_role()
region = sess.boto_session.region_name
bucket = sess.default_bucket()
prefix = "sagemaker/titanic"

print(f"Role: {role}")
print(f"Region: {region}")
print(f"Bucket: {bucket}")

## 2. Subir datos crudos a S3

In [ ]:
raw_data_s3uri = sess.upload_data(
    path="data/titanic_raw.csv",
    bucket=bucket,
    key_prefix=f"{prefix}/raw",
)
print(f"Raw data uploaded to: {raw_data_s3uri}")

## 3. Processing Job

Ejecuta `src/preprocessing.py` en un contenedor sklearn gestionado por SageMaker.

**Entrada**: `titanic_raw.csv` desde S3  
**Salidas**: `train.csv` y `validation.csv` en S3 (sin cabecera, primera columna = `Survived`)

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="titanic-processing",
)

sklearn_processor.run(
    code="src/preprocessing.py",
    inputs=[
        ProcessingInput(
            source=raw_data_s3uri,
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train",
            destination=f"s3://{bucket}/{prefix}/processed/train",
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/output/validation",
            destination=f"s3://{bucket}/{prefix}/processed/validation",
        ),
    ],
    wait=True,
)

train_s3uri = f"s3://{bucket}/{prefix}/processed/train"
validation_s3uri = f"s3://{bucket}/{prefix}/processed/validation"
print(f"Train data: {train_s3uri}")
print(f"Validation data: {validation_s3uri}")

## 4. Training Job

Entrena un `RandomForestClassifier` usando `src/train.py` como script personalizado.  
Las métricas (accuracy, precision, recall, F1) se capturan desde los logs del contenedor.

In [ ]:
from sagemaker.sklearn.estimator import SKLearn

sklearn_estimator = SKLearn(
    entry_point="train.py",
    source_dir="src",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    base_job_name="titanic-training",
    hyperparameters={
        "n_estimators": 100,
        "max_depth": 8,
        "random_state": 42,
    },
    metric_definitions=[
        {"Name": "accuracy",  "Regex": "Accuracy: ([0-9.]+)"},
        {"Name": "precision", "Regex": "Precision: ([0-9.]+)"},
        {"Name": "recall",    "Regex": "Recall: ([0-9.]+)"},
        {"Name": "f1",        "Regex": "F1: ([0-9.]+)"},
    ],
    max_run=20 * 60,
    use_spot_instances=True,
    max_wait=30 * 60,
)

sklearn_estimator.fit(
    {
        "train": train_s3uri,
        "validation": validation_s3uri,
    },
    wait=True,
)

## 5. Resultados

Recupera el nombre del job, la URI del modelo en S3 y las métricas capturadas.

In [ ]:
job_desc = sklearn_estimator.latest_training_job.describe()
model_s3uri = job_desc["ModelArtifacts"]["S3ModelArtifacts"]
job_name = job_desc["TrainingJobName"]

print(f"Job name:       {job_name}")
print(f"Model artifact: {model_s3uri}")
print()

sm_client = boto3.client("sagemaker")
metrics = sm_client.describe_training_job(TrainingJobName=job_name).get("FinalMetricDataList", [])
print("Metrics captured by SageMaker:")
for m in metrics:
    print(f"  {m['MetricName']}: {m['Value']:.4f}")